# 02 · Explore the NSW electricity market

Prices, operational demand, rooftop solar and Sydney temperature. Every analysis uses the same cleaned artifact as the dashboards. Chart aggregation is for presentation; summary metrics use half-hours. Negative prices and extremes remain in the analysis.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'nem_analysis.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from nem_analysis import load_market, filter_market, aggregate_market, smooth_prices, price_episodes, score_forecasts, season_labels
market, metadata = load_market()
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = ['#087e72', '#df9562', '#8296b0', '#233c35']
print(f"{len(market):,} half-hour positions | {market.index.min()}–{market.index.max()} | AEST (UTC+10)")

44,911 half-hour positions | 2022-01-01 00:00:00–2024-07-24 15:00:00 | AEST (UTC+10)


## Price distributions and Australian seasons

In [2]:
explore = market.assign(season=season_labels(market.index), year=market.index.year)
seasonal = explore.groupby('season').agg(observations=('price','count'), mean_price=('price','mean'), median_price=('price','median'), peak_price=('price','max'), mean_demand_MW=('demand','mean'), mean_solar_MW=('solar','mean'))
display(seasonal.reindex(['Summer','Autumn','Winter','Spring']).round(2))
display(explore.groupby('year').price.agg(['count','mean','median','min','max']).round(2))

,observations,mean_price,median_price,peak_price,mean_demand_MW,mean_solar_MW
season,,,,,,
Summer,11517,85.81,77.75,13325.68,7536.29,995.61
Autumn,13248,162.86,107.34,16599.91,7491.20,692.56
Winter,11406,185.19,121.83,15100.00,8570.63,559.79
Spring,8736,102.02,89.15,6020.96,7011.04,963.62


,count,mean,median,min,max
year,,,,,
2022,17519,182.72,135.07,-207.78,15100.00
2023,17517,95.94,83.67,-517.32,9805.30
2024,9871,128.45,79.67,-515.08,16599.91


## Price, demand and solar over time
Daily means can conceal short extremes. Use the dashboard’s five-minute peak overlay and event explorer to inspect them.

In [3]:
from plotly.subplots import make_subplots
daily, _ = aggregate_market(market, 'Daily')
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=.13, subplot_titles=['Spot price · daily average', 'Demand and rooftop solar · daily average'])
fig.add_trace(go.Scatter(x=daily.index, y=daily.price, name='Spot price', line_color='#087e72'),row=1,col=1)
for column, label, color in [('demand','Operational demand','#233c35'),('solar','Rooftop solar','#df9562')]:
    fig.add_trace(go.Scatter(x=daily.index,y=daily[column],name=label,line_color=color),row=2,col=1)
fig.update_yaxes(title_text='AUD/MWh',row=1,col=1);fig.update_yaxes(title_text='MW',row=2,col=1)
fig.update_xaxes(title_text='AEST (UTC+10)',row=2,col=1)
fig.update_layout(template='plotly_white',height=650,hovermode='x unified')
fig.show(renderer='plotly_mimetype')

## The daily rhythm
Hourly price means include extremes. This is an association, not evidence that the hour itself causes high prices.

In [4]:
profile = market.groupby(market.index.hour).agg(price=('price','mean'), demand=('demand','mean'), solar=('solar','mean'))
fig = px.bar(profile.reset_index(),x='time',y='price',labels={'time':'Hour · AEST','price':'Mean AUD/MWh'},title='Average price by hour')
fig.show(renderer='plotly_mimetype')
display(profile.round(2))

,price,demand,solar
time,,,
0,133.58,7638.19,0.00
1,126.03,7247.88,0.00
2,117.49,6836.20,0.00
3,112.92,6580.23,0.00
4,115.14,6570.12,0.00
5,127.23,6844.65,8.64
6,134.77,7381.53,121.93
7,158.08,7891.25,485.62
8,114.26,7908.59,1127.94


## Recorded demand forecasts and weather
The recorded POE50 forecast is descriptive only: issue times are missing, so it is excluded from the predictive feature set. Temperature correlations do not establish a causal relationship.

In [5]:
demand_error = (market.forecastDemand - market.demand).dropna()
display(pd.Series({'recorded_demand_forecast_MAE_MW': demand_error.abs().mean(), 'recorded_demand_forecast_bias_MW': demand_error.mean(), 'negative_price_share': market.price.dropna().lt(0).mean(), 'half_hour_spike_share_at_300': market.price.dropna().ge(300).mean()}).to_frame('value'))
display(market[['price','demand','solar','temperature']].corr().round(3))

,value
recorded_demand_forecast_MAE_MW,51.231458
recorded_demand_forecast_bias_MW,-3.125426
negative_price_share,0.041530
half_hour_spike_share_at_300,0.071503


,price,demand,solar,temperature
price,1.000,0.234,-0.146,-0.124
demand,0.234,1.000,-0.395,-0.232
solar,-0.146,-0.395,1.000,0.486
temperature,-0.124,-0.232,0.486,1.000


## Continue
[03 · Forecasting benchmarks](03_forecasting.ipynb) separates exploratory relationships from features available before a forecast is made.